In [1]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(
            f"GPU {i}: {torch.cuda.get_device_name(i)} | "
            f"VRAM: {props.total_memory / 1024**3:.2f} GB"
        )

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4 | VRAM: 14.56 GB
GPU 1: Tesla T4 | VRAM: 14.56 GB


In [2]:
import requests

response = requests.get(
    "https://huggingface.co",
    timeout=10
)

print("Internet status:", response.status_code)

Internet status: 200


In [3]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        print(p)

/kaggle/input/datasets/hassanch6138/localsql-phase3-input/manifest.jsonl
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/PROJECT.md
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/.gitignore
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/pyproject.toml
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/README.md
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/uv.lock
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/.python-version
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/CLAUDE.md
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/tests/__init__.py
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/localsql-phase3-src/PROJECT.md
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/localsql-phase3-s

In [4]:
from pathlib import Path
import shutil

SOURCE_DIR = Path(
    "/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src"
)

MANIFEST_SOURCE = Path(
    "/kaggle/input/datasets/hassanch6138/localsql-phase3-input/manifest.jsonl"
)

PROJECT_DIR = Path("/kaggle/working/localsql")

# Clean old working copy if this cell is rerun
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

# Copy source, but ignore the accidental nested duplicate repo
shutil.copytree(
    SOURCE_DIR,
    PROJECT_DIR,
    ignore=shutil.ignore_patterns("localsql-phase3-src")
)

print("Project copied to:", PROJECT_DIR)
print("Manifest source:", MANIFEST_SOURCE)

print("\nTop-level project files:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" -", p.name)

Project copied to: /kaggle/working/localsql
Manifest source: /kaggle/input/datasets/hassanch6138/localsql-phase3-input/manifest.jsonl

Top-level project files:
 - .gitignore
 - .python-version
 - CLAUDE.md
 - PROJECT.md
 - README.md
 - configs
 - data
 - docs
 - pyproject.toml
 - scripts
 - src
 - tests
 - uv.lock


In [5]:
MANIFEST_DEST = (
    PROJECT_DIR
    / "data"
    / "benchmarks"
    / "bird_mini_dev"
    / "generation"
    / "manifest.jsonl"
)

MANIFEST_DEST.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(MANIFEST_SOURCE, MANIFEST_DEST)

print("Manifest copied to:")
print(MANIFEST_DEST)
print("Exists:", MANIFEST_DEST.exists())
print("Size:", MANIFEST_DEST.stat().st_size, "bytes")

Manifest copied to:
/kaggle/working/localsql/data/benchmarks/bird_mini_dev/generation/manifest.jsonl
Exists: True
Size: 4223357 bytes


In [6]:
import json

count = 0
first_example = None

with open(MANIFEST_DEST, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)

            if first_example is None:
                first_example = row

            count += 1

print("Total examples:", count)
print("First example ID:", first_example["example_id"])
print("First database:", first_example["db_id"])
print("\nFirst question:")
print(first_example["question"])

Total examples: 500
First example ID: bird-mini-dev-sqlite-0000
First database: debit_card_specializing

First question:
What is the ratio of customers who pay in EUR against customers who pay in CZK?


In [7]:
forbidden_fields = {
    "sql",
    "gold_sql",
    "target_sql",
    "completion",
    "reference_sql",
}

found_forbidden = forbidden_fields.intersection(first_example.keys())

print("Fields in first example:")
print(sorted(first_example.keys()))

assert not found_forbidden, (
    f"Gold leakage detected: {found_forbidden}"
)

print("\nGold leakage check: PASSED")

Fields in first example:
['business_context', 'db_id', 'dialect', 'difficulty', 'example_id', 'prompt', 'question', 'serialized_schema']

Gold leakage check: PASSED


In [8]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [9]:
!pwd
!ls

/kaggle/working/localsql
CLAUDE.md  data  PROJECT.md	 README.md  src    uv.lock
configs    docs  pyproject.toml  scripts    tests


In [10]:
import importlib.util
import torch

packages = [
    "transformers",
    "accelerate",
    "bitsandbytes",
]

print("Torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

for pkg in packages:
    print(
        f"{pkg}:",
        "installed"
        if importlib.util.find_spec(pkg)
        else "NOT installed"
    )

Torch: 2.10.0+cu128
CUDA runtime: 12.8
CUDA available: True
transformers: installed
accelerate: installed
bitsandbytes: NOT installed


In [11]:
!pip install -q -e .
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for localsql (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 93.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.1 MB/s eta 0:00:00:00:01


In [12]:
import torch
import transformers
import accelerate
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

torch: 2.10.0+cu128
transformers: 5.17.0
accelerate: 1.15.0
bitsandbytes: 0.50.2
CUDA: 12.8
CUDA available: True


In [13]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("Using physical GPU:", os.environ["CUDA_VISIBLE_DEVICES"])

Using physical GPU: 0


In [14]:
!python scripts/run_baseline.py \
    --manifest data/benchmarks/bird_mini_dev/generation/manifest.jsonl \
    --run-id kaggle-dry-run \
    --dry-run \
    --limit 5

Manifest: 5 gold-free examples parsed OK (no gold fields present).
Manifest sha256: 6903e95f373e267d5ebbbf5fa9b47100c3cf3ded9294595e1c5a026dbe0befa7
Run directory: /kaggle/working/localsql/data/runs/kaggle-dry-run
Already completed (resume): 0 / 5
Remaining to generate: 5
Dry run OK -- no model loaded, no CUDA required.


In [15]:
!python scripts/run_baseline.py \
    --manifest data/benchmarks/bird_mini_dev/generation/manifest.jsonl \
    --run-id qwen3-4b-base-token-profile \
    --token-profile

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 2.36MB/s]
tokenizer_config.json: 9.38kB [00:00, 24.8MB/s]
vocab.json: 2.78MB [00:00, 111MB/s]
merges.txt: 1.67MB [00:00, 110MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 20.9MB/s]
Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Counting prompt tokens ...
{
  "context_mode": "with_business_context",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 500,
  "prompt_token_stats": {
    "count": 500,
    "min": 2.0,
    "median": 2.0,
    "p90": 2.0,
    "p95": 2.0,
    "max": 2.0
  },
  "count_above_4096": 0,
  "count_above_8192": 0
}

Wrote /kaggle/working/localsql/data/runs/qwen3-4b-base-token-profile/token_profile.json


In [16]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
TOKENIZER_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=TOKENIZER_REVISION,
)

sample_prompt = first_example["prompt"]

messages = [
    {
        "role": "user",
        "content": sample_prompt,
    }
]

encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

print("Returned object type:", type(encoded))
print("Keys:", encoded.keys())
print("input_ids shape:", encoded["input_ids"].shape)
print("attention_mask shape:", encoded["attention_mask"].shape)

print()
print("WRONG way - len(encoded):", len(encoded))
print(
    "CORRECT token count:",
    encoded["input_ids"].shape[-1]
)

Returned object type: <class 'transformers.tokenization_utils_base.BatchEncoding'>
Keys: KeysView({'input_ids': tensor([[151644,    872,    198,  46487,    510,   2610,    525,    264,   1467,
           4686,   6222,   3588,   1614,    624,  31115,   6896,    825,   1349,
          15382,   7870,   3239,    429,  11253,    279,   3405,   1667,   1172,
            279,   3897,   4625,  10802,    624,   5598,   7870,   1172,    624,
           5404,    537,    990,  50494,    382,     35,   5863,   3965,    510,
          37042,    271,   3540,  35839,    510,  40154,   1006,    220,  12277,
            915,  30381,  24826,    345,    220,  37103,  15762,    345,    220,
          28453,  15762,    198,    692,  39525,  74628,   1006,    220,  20854,
          19765,    915,  30381,  24826,    345,    220,  28525,    915,  30381,
            345,    220,  14106,  15762,    345,    220,  37103,  15762,    198,
            692,  10144,   1006,    220,   5643,    915,  30381,  24826,    34

In [17]:
import json
import numpy as np

token_counts = []

with open(MANIFEST_DEST, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        row = json.loads(line)

        messages = [
            {
                "role": "user",
                "content": row["prompt"],
            }
        ]

        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors=None,
        )

        token_count = len(encoded["input_ids"])
        token_counts.append(token_count)

token_counts = np.array(token_counts)

print("Examples:", len(token_counts))
print("Minimum:", int(token_counts.min()))
print("Median:", int(np.median(token_counts)))
print("P90:", int(np.percentile(token_counts, 90)))
print("P95:", int(np.percentile(token_counts, 95)))
print("Maximum:", int(token_counts.max()))
print("Above 4096:", int((token_counts > 4096).sum()))
print("Above 8192:", int((token_counts > 8192).sum()))

Examples: 500
Minimum: 203
Median: 930
P90: 2322
P95: 2368
Maximum: 2428
Above 4096: 0
Above 8192: 0


In [20]:
from pathlib import Path

backend_path = Path(
    "/kaggle/working/localsql/src/localsql/model/qwen_backend.py"
)

text = backend_path.read_text(encoding="utf-8")

# 1. Update the import
old_import = (
    "from localsql.model.generation import "
    "BackendGenerationResult, build_model_inputs"
)

new_import = (
    "from localsql.model.generation import "
    "BackendGenerationResult, build_model_inputs, count_input_tokens"
)

if old_import in text:
    text = text.replace(old_import, new_import, 1)
    print("Updated generation import.")
elif "count_input_tokens" in text:
    print("Import already contains count_input_tokens.")
else:
    raise RuntimeError("Could not find expected generation import.")

# 2. Replace the real function we just inspected
old_function = '''    def count_prompt_tokens(self, canonical_prompt: str) -> int:
        """Token count of the fully chat-templated input (what the model
        actually sees), for token-profile mode."""
        input_ids = build_model_inputs(self._tokenizer, canonical_prompt)
        return len(input_ids)
'''

new_function = '''    def count_prompt_tokens(self, canonical_prompt: str) -> int:
        """Token count of the fully chat-templated input (what the model
        actually sees), for token-profile mode."""
        encoded = build_model_inputs(self._tokenizer, canonical_prompt)
        return count_input_tokens(encoded)
'''

if old_function in text:
    text = text.replace(old_function, new_function, 1)
    print("Updated count_prompt_tokens().")
elif "return count_input_tokens(encoded)" in text:
    print("count_prompt_tokens() already patched.")
else:
    raise RuntimeError(
        "Could not find the expected count_prompt_tokens() function."
    )

backend_path.write_text(text, encoding="utf-8")

print("\nPatch completed successfully.")

Updated generation import.
Updated count_prompt_tokens().

Patch completed successfully.


In [21]:
text = backend_path.read_text(encoding="utf-8")
lines = text.splitlines()

for i, line in enumerate(lines):
    if "count_prompt_tokens" in line:
        for j in range(max(0, i - 3), min(len(lines), i + 10)):
            print(f"{j+1:03}: {lines[j]}")
        break

115:         self._tokenizer = AutoTokenizer.from_pretrained(cfg.model.id, revision=resolved_revision)
116:         return resolved_revision
117: 
118:     def count_prompt_tokens(self, canonical_prompt: str) -> int:
119:         """Token count of the fully chat-templated input (what the model
120:         actually sees), for token-profile mode."""
121:         encoded = build_model_inputs(self._tokenizer, canonical_prompt)
122:         return count_input_tokens(encoded)
123: 
124:     def generate_one(self, prompt: str) -> BackendGenerationResult:
125:         import torch
126: 
127:         cfg = self.config


In [22]:
!python scripts/run_baseline.py \
    --manifest data/benchmarks/bird_mini_dev/generation/manifest.jsonl \
    --run-id qwen3-4b-base-token-profile-fixed \
    --token-profile


Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Counting prompt tokens ...
{
  "context_mode": "with_business_context",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 500,
  "prompt_token_stats": {
    "count": 500,
    "min": 203.0,
    "median": 930.5,
    "p90": 2322.7,
    "p95": 2368.05,
    "max": 2428.0
  },
  "count_above_4096": 0,
  "count_above_8192": 0
}

Wrote /kaggle/working/localsql/data/runs/qwen3-4b-base-token-profile-fixed/token_profile.json


In [23]:
import os

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

CUDA_VISIBLE_DEVICES: 0


In [24]:
!nvidia-smi

Tue Sep 15 11:46:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:
!python scripts/run_baseline.py \
    --manifest data/benchmarks/bird_mini_dev/generation/manifest.jsonl \
    --run-id qwen3-4b-base-nf4-smoke \
    --limit 5

Loading Qwen/Qwen3-4B-Instruct-2507 (4-bit nf4) ...
model.safetensors.index.json: 32.8kB [00:00, 62.8MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:34<00:00, 11.36s/it]
Download complete: 100%|████████████████████| 8.04G/8.04G [00:34<00:00, 236MB/s]
generation_config.json: 100%|██████████████████| 238/238 [00:00<00:00, 1.21MB/s]
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Resuming: 0 / 5 already completed.
  ERROR bird-mini-dev-sqlite-0000: AttributeError: 
  ERROR bird-mini-dev-sqlite-0001: AttributeError: 
  ERROR bird-mini-dev-sqlite-0002: AttributeError: 
  ERROR bird-mini-dev-sqlite-0003: AttributeError: 
  ERROR bird-mini-dev-sqlite-0004: AttributeError: 

Generated 0 (failed 5) this run. Completed overall: 0/5
Summary: /kaggle/working/localsql/data/runs/qwen3-4b-base-nf4-smoke/summary.json
Predictions (Phase 2 contract): /kaggle/working/localsql/data/runs/qwen3-4b-base-nf4-smoke/predictions.jsonl


In [30]:
!python scripts/run_baseline.py \
    --manifest data/benchmarks/bird_mini_dev/generation/manifest.jsonl \
    --run-id qwen3-4b-base-nf4-smoke-v2 \
    --limit 5

Loading Qwen/Qwen3-4B-Instruct-2507 (4-bit nf4) ...
Loading weights: 100%|███████████████████████| 398/398 [00:02<00:00, 146.92it/s]
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Resuming: 0 / 5 already completed.

Generated 5 (failed 0) this run. Completed overall: 5/5
Summary: /kaggle/working/localsql/data/runs/qwen3-4b-base-nf4-smoke-v2/summary.json
Predictions (Phase 2 contract): /kaggle/working/localsql/data/runs/qwen3-4b-base-nf4-smoke-v2/predictions.jsonl


In [31]:
from pathlib import Path
import json

RUN_DIR = Path(
    "/kaggle/working/localsql/data/runs/"
    "qwen3-4b-base-nf4-smoke-v2"
)

print("FILES:")
for p in sorted(RUN_DIR.iterdir()):
    print(" -", p.name)

print("\nSUMMARY:")
with open(RUN_DIR / "summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2))

FILES:
 - generations.jsonl
 - predictions.jsonl
 - run_config.json
 - summary.json

SUMMARY:
{
  "run_id": "qwen3-4b-base-nf4-smoke-v2",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "generation_config": {
    "do_sample": false,
    "max_new_tokens": 512,
    "seed": 42
  },
  "context_mode": "with_business_context",
  "manifest_sha256": "6903e95f373e267d5ebbbf5fa9b47100c3cf3ded9294595e1c5a026dbe0befa7",
  "requested_examples": 5,
  "completed_examples": 5,
  "generated_this_run": 5,
  "failed_this_run": 0,
  "stopped_early_oom": false,
  "total_runtime_seconds": 31.5,
  "latency_ms_stats": {
    "count": 5,
    "min": 5056.2453610000375,
    "median": 5737.173149999762,
    "p90": 7835.99,
    "p95": 7983.8,
    "max": 81

In [32]:
import json

with open(
    RUN_DIR / "generations.jsonl",
    "r",
    encoding="utf-8"
) as f:
    rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Generation count:", len(rows))
print("Fields:", sorted(rows[0].keys()))

for i, row in enumerate(rows, start=1):
    print("\n" + "=" * 90)
    print(f"EXAMPLE {i}")
    print("=" * 90)

    print("Example ID:", row.get("example_id"))
    print("DB:", row.get("db_id"))
    print("Input tokens:", row.get("input_tokens"))
    print("Output tokens:", row.get("output_tokens"))
    print("Latency ms:", row.get("latency_ms"))

    print("\nRAW COMPLETION:")
    print(row.get("raw_completion"))

    print("\nPREDICTED SQL:")
    print(row.get("predicted_sql"))

Generation count: 5
Fields: ['context_mode', 'db_id', 'error', 'example_id', 'input_tokens', 'is_oom', 'latency_ms', 'model_revision', 'output_tokens', 'predicted_sql', 'raw_completion', 'status']

EXAMPLE 1
Example ID: bird-mini-dev-sqlite-0000
DB: debit_card_specializing
Input tokens: 239
Output tokens: 51
Latency ms: 5056.2453610000375

RAW COMPLETION:
SELECT 
  COUNT(CASE WHEN c.Currency = 'EUR' THEN 1 END) * 1.0 / 
  COUNT(CASE WHEN c.Currency = 'CZK' THEN 1 END) AS ratio
FROM customers c;

PREDICTED SQL:
SELECT 
  COUNT(CASE WHEN c.Currency = 'EUR' THEN 1 END) * 1.0 / 
  COUNT(CASE WHEN c.Currency = 'CZK' THEN 1 END) AS ratio
FROM customers c;

EXAMPLE 2
Example ID: bird-mini-dev-sqlite-0001
DB: debit_card_specializing
Input tokens: 249
Output tokens: 70
Latency ms: 5737.173149999762

RAW COMPLETION:
SELECT c.CustomerID
FROM yearmonth ym
JOIN customers c ON ym.CustomerID = c.CustomerID
WHERE c.Segment = 'LAM'
  AND ym.Date BETWEEN '201201' AND '201212'
ORDER BY ym.Consumption ASC

In [33]:
import os
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

CUDA_VISIBLE_DEVICES: 0


In [34]:
!python scripts/run_baseline.py \
    --manifest data/benchmarks/bird_mini_dev/generation/manifest.jsonl \
    --run-id qwen3-4b-base-nf4

Loading Qwen/Qwen3-4B-Instruct-2507 (4-bit nf4) ...
Loading weights: 100%|███████████████████████| 398/398 [00:02<00:00, 144.80it/s]
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Resuming: 0 / 500 already completed.

Generated 500 (failed 0) this run. Completed overall: 500/500
Summary: /kaggle/working/localsql/data/runs/qwen3-4b-base-nf4/summary.json
Predictions (Phase 2 contract): /kaggle/working/localsql/data/runs/qwen3-4b-base-nf4/predictions.jsonl


In [35]:
from pathlib import Path
import json

FULL_RUN_DIR = Path(
    "/kaggle/working/localsql/data/runs/"
    "qwen3-4b-base-nf4"
)

with open(
    FULL_RUN_DIR / "summary.json",
    "r",
    encoding="utf-8"
) as f:
    full_summary = json.load(f)

print(json.dumps(full_summary, indent=2))

{
  "run_id": "qwen3-4b-base-nf4",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "generation_config": {
    "do_sample": false,
    "max_new_tokens": 512,
    "seed": 42
  },
  "context_mode": "with_business_context",
  "manifest_sha256": "6903e95f373e267d5ebbbf5fa9b47100c3cf3ded9294595e1c5a026dbe0befa7",
  "requested_examples": 500,
  "completed_examples": 500,
  "generated_this_run": 500,
  "failed_this_run": 0,
  "stopped_early_oom": false,
  "total_runtime_seconds": 3045.86,
  "latency_ms_stats": {
    "count": 500,
    "min": 1249.9183289992288,
    "median": 5576.758827499361,
    "p90": 8890.7,
    "p95": 9981.84,
    "max": 34239.53407999943
  },
  "input_token_stats": {
    "count": 500,
    "min": 203.0,
    "media

In [36]:
def count_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

print(
    "Generations:",
    count_jsonl(FULL_RUN_DIR / "generations.jsonl")
)

print(
    "Predictions:",
    count_jsonl(FULL_RUN_DIR / "predictions.jsonl")
)

Generations: 500
Predictions: 500


In [37]:
import json

rows = []

with open(
    FULL_RUN_DIR / "generations.jsonl",
    "r",
    encoding="utf-8"
) as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

successes = [
    r for r in rows
    if r.get("status") == "success"
]

markdown_fences = sum(
    "```" in (r.get("raw_completion") or "")
    for r in successes
)

starts_with_select = sum(
    (r.get("predicted_sql") or "")
    .lstrip()
    .upper()
    .startswith(("SELECT", "WITH"))
    for r in successes
)

hit_512 = sum(
    r.get("output_tokens") == 512
    for r in successes
)

print("Successful generations:", len(successes))
print("Contains markdown fences:", markdown_fences)
print("Starts with SELECT/WITH:", starts_with_select)
print("Hit 512-token ceiling:", hit_512)

Successful generations: 0
Contains markdown fences: 0
Starts with SELECT/WITH: 0
Hit 512-token ceiling: 0


In [38]:
from pathlib import Path
import shutil

EXPORT_DIR = Path(
    "/kaggle/working/localsql-phase3-baseline-export"
)

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

EXPORT_DIR.mkdir(parents=True)

# Full canonical baseline
shutil.copytree(
    FULL_RUN_DIR,
    EXPORT_DIR / "qwen3-4b-base-nf4"
)

# Correct token profile
shutil.copytree(
    Path(
        "/kaggle/working/localsql/data/runs/"
        "qwen3-4b-base-token-profile-fixed"
    ),
    EXPORT_DIR / "qwen3-4b-base-token-profile-fixed"
)

# Exact source files used after the Kaggle fixes
(EXPORT_DIR / "source-used").mkdir()

for src in [
    Path("/kaggle/working/localsql/src/localsql/model/generation.py"),
    Path("/kaggle/working/localsql/src/localsql/model/qwen_backend.py"),
    Path("/kaggle/working/localsql/configs/model.yaml"),
]:
    shutil.copy2(
        src,
        EXPORT_DIR / "source-used" / src.name
    )

archive = shutil.make_archive(
    "/kaggle/working/localsql-phase3-baseline-export",
    "zip",
    EXPORT_DIR,
)

print("Created:")
print(archive)

Created:
/kaggle/working/localsql-phase3-baseline-export.zip
